# Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết

Phần 1 – Bài 2 (trang 28): Khảo sát và tiền xử lý dữ liệu Bank marketing giống phần hướng dẫn.

Dữ liệu dùng: `bank-full.csv` (45.211 khách hàng, 16 thuộc tính + thuộc tính quyết định `y` – có đăng ký gửi tiền có kỳ hạn hay không).

In [1]:
SV = "Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT"
DATA = "../../data/"  # thư mục chứa dữ liệu
print(SV)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [2]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

### Chuẩn bị: thực hiện lại các bước của các câu trước

In [3]:
df = pd.read_csv(DATA + 'bank-full.csv', sep=';')
df['y_num'] = (df['y'] == 'yes').astype(int)   # dùng để tính tỷ lệ đăng ký

---
## Thực hiện yêu cầu

In [4]:
print(SV)
# 'unknown' chính là giá trị bị thiếu
df = df.replace('unknown', np.nan)

# Xóa các thuộc tính không cần thiết
#  - poutcome: thiếu ~82%
#  - contact : thiếu ~29%, chỉ là kênh liên lạc
#  - day, month: ngày/tháng gọi điện, không phải đặc điểm khách hàng
#  - pdays: -1 cho khách chưa liên hệ, thông tin đã có trong 'previous'
df = df.drop(['poutcome', 'contact', 'day', 'month', 'pdays'], axis=1)

# job, education (định tính) -> điền bằng mode theo cùng thuộc tính quyết định y
for col in ['job', 'education']:
    df[col] = df[col].fillna(df.groupby('y')[col].transform(lambda s: s.mode()[0]))

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [5]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
duration     0
campaign     0
previous     0
y            0
y_num        0
dtype: int64

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   duration   45211 non-null  int64
 9   campaign   45211 non-null  int64
 10  previous   45211 non-null  int64
 11  y          45211 non-null  str  
 12  y_num      45211 non-null  int64
dtypes: int64(6), str(7)
memory usage: 4.5 MB


**Nhận xét:**

- Đã đổi `'unknown'` thành NaN; `job` và `education` được điền bằng mode của nhóm khách có cùng giá trị `y` (giống cách điền theo cùng thuộc tính quyết định trong hướng dẫn).
- Đã xóa `poutcome` (thiếu ≈82%), `contact` (thiếu ≈29%, chỉ là kênh liên lạc), `day`, `month` (thời điểm gọi, không phải đặc điểm khách hàng) và `pdays` (thông tin trùng với `previous`).
- Sau tiền xử lý không còn giá trị rỗng; dữ liệu còn 12 thuộc tính (cộng cột phụ `y_num`).